# 3d_uretim_modeli — Seçilen Yapının Bütün Veriyle Yeniden Eğitimi

**Neden:** `7_canli_tr`'nin E3 analizi bir rejim değişimi gösterdi:
- 2021-2025'te resmî plan sıcak günlerde (≥27°C) tüketimi sistematik olarak düşük tahmin ediyordu (−%2,2 ile −%4,1 arası). Model bunu öğrenip yukarı düzeltiyordu.
- 2025-26'da plan sıcak günlerde isabetli (−%0,08). Modelin öğrendiği düzeltme bu yüzden fazla kalıyor (+%1,36).

Model, 2025-26 verisini hiç görmedi; o yıl test yılıydı.

**Ne değişmiyor:**
- Seçilen model ve iki yedeği.
- Özellikler, hiperparametreler, hedef türleri.
- Seçim yeniden yapılmıyor.

**Ne değişiyor:**
1. **Veri:** Test yılı dahil, bütün özelliklerin mevcut olduğu son güne kadar.
2. **Ağaç sayısı sabitleniyor:** `3c`'deki kural erken durdurma için eğitimin son 6 ayını ayırıyordu. Bu kural burada uygulansaydı, son 6 ay (2026 yazı) eğitime girmez ve model tam da öğrenmesi gereken dönemi göremezdi. Bu yüzden her model için ağaç sayısı, `3c`'nin erken durdurmayla bulduğu değerde (kayıtlı modelin `best_iteration + 1` değeri) sabitleniyor, ve model bütün veriyle erken durdurma olmadan eğitiliyor. Doğrulamada bulunan tur sayısıyla bütün veriye yeniden fit etmek yaygın bir yöntem.
3. **Bazlar ve sapma tablosu:** Bütün veriden, ve son 12 aydan yeniden hesaplanıyor.

**Değerlendirme hakkında dürüst bir not:** Yeni modeller test yılını da gördüğü için, ayrı tutulmuş bir test verisi kalmadı. E bölümündeki karşılaştırma **eğitim içi**; yalnızca beklenen yöndeki değişikliğin gerçekleşip gerçekleşmediğini kontrol ediyor, bir başarı ölçüsü değil. Gerçek değerlendirme, `7_canli_tr`'nin D bölümündeki günlük canlı karşılaştırmayla yapılacak.

Belirsizlik bandı (`tr_gunoncesi_q90.json`) değiştirilmiyor. Çapraz doğrulamadan hesaplanmıştı ve yeni modeller için ayrı tutulmuş veri yok. Kapsama oranı, D bölümünde canlı olarak izleniyor.

**Kayıt:** Eski model dosyaları `agent/arsiv/<zaman>/` klasörüne taşınıyor; geri dönmek için dosyaları geri kopyalamak yeterli. `7_canli_tr` yeni modelleri otomatik olarak kullanıyor.

## A. Veri ve Özellikler (`3c` ile aynı kurallar)

In [1]:
import os
import json
import time
import shutil
import numpy as np
import pandas as pd
import xgboost as xgb

KOK = r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje_Bonus"
np.random.seed(42)

df = pd.read_csv(os.path.join(KOK, "data", "tr_ozellikli_saatlik.csv"))
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
lp = pd.read_csv(os.path.join(KOK, "data", "tr_yuk_plani_saatlik.csv"))
lp["date"] = pd.to_datetime(lp["date"])
df = df.merge(lp[["date", "lep"]], on="date", how="left")
df["gun_t"] = df["date"].dt.tz_localize(None).dt.normalize()

# Resmi plan temizligi (3c Revizyon 2): imkansiz degerler ve gun ici duz plan
oran = df["consumption"] / df["lep"]
sorun = df["lep"].notna() & ((df["lep"] <= 0) | (oran > 2) | (oran < 0.5))
df.loc[sorun, "lep"] = np.nan
gs = df.groupby("gun_t")["lep"].agg(["std", "mean", "count"])
duz = gs.index[(gs["count"] == 24) & (gs["std"] < 0.01 * gs["mean"])]
df.loc[df["gun_t"].isin(duz), "lep"] = np.nan
print("Temizlenen plan saati:", int(sorun.sum()), "| duz plan gunu:", [str(g.date()) for g in duz])

df["lag_48"] = df["consumption"].shift(48)
df["lag_72"] = df["consumption"].shift(72)
df["rol_24_48"] = df["consumption"].shift(48).rolling(24).mean()
df["rol_168_48"] = df["consumption"].shift(48).rolling(168).mean()
df["tavg_dun"] = df["tavg_c"].shift(24)
df["lep_hata"] = df["lep"] - df["consumption"]
df["lep_hata_48"] = df["lep_hata"].shift(48)
df["lep_hata_168"] = df["lep_hata"].shift(168)
df["lep_hata_ort_7g"] = df["lep_hata"].shift(48).rolling(168, min_periods=120).mean()
df["lep_oran_hedef"] = df["consumption"] / df["lep"] - 1

ESKI = {}
for anahtar, dosya in [("secilen", "tr_xgb_gunoncesi_secilen"), ("havasiz", "tr_xgb_gunoncesi_yedek_havasiz"),
                       ("plansiz", "tr_xgb_gunoncesi_yedek_plansiz")]:
    m = xgb.XGBRegressor(); m.load_model(os.path.join(KOK, "agent", dosya + ".json"))
    with open(os.path.join(KOK, "agent", dosya + "_meta.json"), encoding="utf-8") as f:
        ESKI[anahtar] = {"model": m, "meta": json.load(f), "dosya": dosya}
AGIRLIK = ESKI["secilen"]["meta"].get("sehir_agirliklari") or {}

def seri(dosya):
    s = pd.read_csv(os.path.join(KOK, "data", dosya))
    return pd.Series(s["deger"].values, index=pd.to_datetime(s["date"]))
idx = pd.DatetimeIndex(df["date"])
if AGIRLIK:
    w = pd.Series(AGIRLIK)
    df["om_era5"] = (pd.DataFrame({s: seri(f"om_era5_{s}.csv") for s in AGIRLIK}) * w).sum(axis=1, min_count=len(w)).reindex(idx).values
    df["om_fc_ham"] = (pd.DataFrame({s: seri(f"om_tahmin_uzun_{s}.csv") for s in AGIRLIK}) * w).sum(axis=1, min_count=len(w)).reindex(idx).values
    df["om_tgun_era5"] = df.groupby("gun_t")["om_era5"].transform("mean")
print("Veri:", df["date"].min(), "->", df["date"].max(), "| modeller:", {k: v["meta"]["model"] for k, v in ESKI.items()})

Temizlenen plan saati: 72 | duz plan gunu: ['2025-04-07']
Veri: 2016-04-01 00:00:00+03:00 -> 2026-09-27 23:00:00+03:00 | modeller: {'secilen': 'Hibrit (oran) + Open-Meteo', 'havasiz': 'Hibrit (oran)', 'plansiz': 'Referans (3_model)'}


## B. Sabit Ağaç Sayıları

Her kayıtlı modelin, `3c`'de erken durdurmayla bulunan en iyi turu okunuyor. Kayıtlı modelde bu bilgi yoksa, modeldeki toplam tur sayısı kullanılıyor ve bu durum yazdırılıyor.

In [2]:
def agac_sayisi(m):
    try:
        bi = m.get_booster().attr("best_iteration")
        if bi is not None:
            return int(bi) + 1, "best_iteration + 1 (3c erken durdurma)"
    except Exception:
        pass
    return int(m.get_booster().num_boosted_rounds()), "modeldeki toplam tur (best_iteration kayitli degil)"

AGAC = {}
for k, v in ESKI.items():
    AGAC[k], kaynak = agac_sayisi(v["model"])
    print(f"{k:8s} ({v['meta']['model']}): {AGAC[k]} agac | kaynak: {kaynak}")

secilen  (Hibrit (oran) + Open-Meteo): 1892 agac | kaynak: best_iteration + 1 (3c erken durdurma)
havasiz  (Hibrit (oran)): 1467 agac | kaynak: best_iteration + 1 (3c erken durdurma)
plansiz  (Referans (3_model)): 422 agac | kaynak: best_iteration + 1 (3c erken durdurma)


## C. Eğitim Kümesi, Bazlar ve Sapma Tablosu

- **Kesim:** Seçilen modelin bütün özelliklerinin mevcut olduğu son günün ertesi günü. Ankara sıcaklığı (NOAA) ve ERA5, Faz 2 verisinin son gününe kadar mevcut.
- **Bazlar:** `3c`'deki yöntemle: normal iş günleri, yıl etkili doğrusal model, R²'yi en yükselten baz sıcaklıklar. Bütün eğitim verisinden hesaplanıyor.
- **Sapma tablosu:** Son 12 ayın (GFS − ERA5) ay × saat ortalaması. Canlı zincir önce son 30 günlük sapmayı kullanıyor; bu tablo onun yedeği.

In [3]:
def baz_sec(d, tcol, bh_aralik, bc_aralik):
    g = d.groupby("gun_t").agg(tuketim=("consumption", "sum"), n=("consumption", "size"), t=(tcol, "first"),
                               tam=("tatil_tam", "max"), yarim=("tatil_yarim", "max"), izin=("idari_izin", "max"),
                               once=("bayram_once_gun", "first"), sonra=("bayram_sonra_gun", "first"))
    g = g[(g.index.dayofweek < 5) & (g["n"] == 24) & (g["tam"] == 0) & (g["yarim"] == 0) & (g["izin"] == 0)
          & (g["once"] > 3) & (g["sonra"] > 3) & g["t"].notna()]
    T, y = g["t"].values, g["tuketim"].values
    yd = pd.get_dummies(g.index.year, drop_first=True).values.astype(float)
    en_iyi = (None, None, -np.inf)
    for bh in bh_aralik:
        for bc in bc_aralik:
            if bc < bh:
                continue
            X = np.column_stack([np.ones(len(y)), np.maximum(0, bh - T), np.maximum(0, T - bc), yd])
            b, *_ = np.linalg.lstsq(X, y, rcond=None)
            r2 = 1 - ((y - X @ b) ** 2).sum() / ((y - y.mean()) ** 2).sum()
            if r2 > en_iyi[2]:
                en_iyi = (float(bh), float(bc), float(r2))
    return en_iyi

TAKVIM = ["saat", "gun", "ay", "hafta_sonu", "tatil_saat", "idari_izin", "bayram_gun_no", "bayram_once_gun", "bayram_sonra_gun"]
gerekli = list(dict.fromkeys(sum([v["meta"]["ozellikler"] for v in ESKI.values()], []) + ["consumption", "lep_oran_hedef"]))
taban = [c for c in gerekli if c not in ["hdd_dun", "cdd_dun", "om_t", "om_tgun", "om_hdd", "om_cdd"]]
if "om_t" in gerekli:
    taban.append("om_era5")
tam_gun = df.dropna(subset=taban).groupby("gun_t").size()
KESIM = (tam_gun[tam_gun == 24].index.max() + pd.Timedelta(days=1))
once = df["gun_t"] < KESIM
BH, BC, R2A = baz_sec(df[once], "tavg_c", np.arange(10, 22.5, 0.5), np.arange(14, 26.5, 0.5))
df["hdd_dun"] = np.maximum(0, BH - df["tavg_dun"])
df["cdd_dun"] = np.maximum(0, df["tavg_dun"] - BC)
SAPMA, OBH, OBC, R2O = {}, None, None, None
if AGIRLIK:
    df["om_t"] = df["om_era5"]                     # egitimde gerceklesen (3c ile ayni)
    df["om_tgun"] = df["om_tgun_era5"]
    OBH, OBC, R2O = baz_sec(df[once], "om_tgun_era5", np.arange(8, 22.5, 0.5), np.arange(14, 28.5, 0.5))
    df["om_hdd"] = np.maximum(0, OBH - df["om_tgun"])
    df["om_cdd"] = np.maximum(0, df["om_tgun"] - OBC)
    m12 = once & (df["gun_t"] >= KESIM - pd.DateOffset(months=12)) & df["om_fc_ham"].notna() & df["om_era5"].notna()
    fk = df.loc[m12, "om_fc_ham"] - df.loc[m12, "om_era5"]
    tab = fk.groupby([df.loc[m12, "date"].dt.month.values, df.loc[m12, "date"].dt.hour.values]).mean()
    SAPMA = {f"{a}-{s}": round(float(v), 4) for (a, s), v in tab.items()}
    print(f"Son 12 ay GFS sapmasi: ort. {fk.mean():+.3f} C ({len(fk)} saat, {len(SAPMA)} ay-saat hucresi)")
print(f"Kesim: {KESIM.date()} (son tam gun {(KESIM - pd.Timedelta(days=1)).date()})")
print(f"Ankara bazlari: HDD {BH} C, CDD {BC} C (R2 {R2A:.4f}) | eski: {ESKI['secilen']['meta']['ankara_bazlari_c']}")
if AGIRLIK:
    print(f"Agirlikli sicaklik bazlari: HDD {OBH} C, CDD {OBC} C (R2 {R2O:.4f}) | eski: {ESKI['secilen']['meta'].get('om_bazlari_c')}")

Son 12 ay GFS sapmasi: ort. +0.517 C (8760 saat, 288 ay-saat hucresi)
Kesim: 2026-09-23 (son tam gun 2026-09-22)
Ankara bazlari: HDD 13.5 C, CDD 17.0 C (R2 0.8601) | eski: [14.0, 17.0]
Agirlikli sicaklik bazlari: HDD 15.5 C, CDD 20.0 C (R2 0.8919) | eski: [15.5, 20.0]


## D. Yeniden Eğitim

`3c` ile aynı hiperparametreler; yalnızca ağaç sayısı sabit ve erken durdurma yok.

In [4]:
XGB_SABIT = dict(learning_rate=0.03, max_depth=8, min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
                 reg_lambda=1.0, tree_method="hist", random_state=42, n_jobs=-1)
YENI = {}
for k, v in ESKI.items():
    meta = v["meta"]
    f, hedef = meta["ozellikler"], meta["hedef"]
    hc = "consumption" if hedef == "seviye" else "lep_oran_hedef"
    e = df[once].dropna(subset=f + [hc])
    e = e[np.isfinite(e[hc])]
    t0 = time.time()
    m = xgb.XGBRegressor(n_estimators=AGAC[k], **XGB_SABIT).fit(e[f], e[hc], verbose=False)
    YENI[k] = m
    print(f"{k:8s}: {len(e):6d} satir ({e['date'].min():%Y-%m} -> {e['date'].max():%Y-%m-%d}) | {AGAC[k]} agac | {time.time() - t0:.0f} sn")

secilen :  91344 satir (2016-04 -> 2026-09-22) | 1892 agac | 11 sn
havasiz :  91344 satir (2016-04 -> 2026-09-22) | 1467 agac | 8 sn
plansiz :  91633 satir (2016-04 -> 2026-09-22) | 422 agac | 2 sn


## E. Eğitim İçi Kontrol (değerlendirme değil)

Eski ve yeni seçilen model, **son 12 ayda** sıcaklık dilimine göre karşılaştırılıyor. Beklenen: Yeni model 2025-26'yı gördüğü için, ≥27°C günlerdeki yukarı sapması küçülmeli. Bu karşılaştırma eğitim verisi üzerinde yapıldığı için yeni model doğal olarak avantajlı; yalnızca değişikliğin beklenen yönde olup olmadığını gösteriyor.

In [5]:
k = "secilen"
meta, f = ESKI[k]["meta"], ESKI[k]["meta"]["ozellikler"]
son12 = df[once & (df["gun_t"] >= KESIM - pd.DateOffset(months=12))].dropna(subset=f + ["consumption", "lep"]).copy()
def tahmin(m, d):
    p = m.predict(d[f])
    return d["lep"].values * (1 + p) if meta["hedef"] == "oran" else p
son12["eski"] = tahmin(ESKI[k]["model"], son12)
son12["yeni"] = tahmin(YENI[k], son12)
tcol = "om_tgun_era5" if "om_tgun_era5" in son12 else "tavg_c"
son12["dilim"] = pd.cut(son12[tcol], [-50, 20, 24, 27, 50], labels=["<20", "20-24", "24-27", ">=27"], right=False)
for c in ["eski", "yeni", "lep"]:
    son12[f"sap_{c}"] = (son12[c] - son12["consumption"]) / son12["consumption"] * 100
kontrol = son12.groupby("dilim", observed=True).agg(saat=("consumption", "size"),
            eski_sapma=("sap_eski", "mean"), yeni_sapma=("sap_yeni", "mean"), plan_sapma=("sap_lep", "mean"),
            eski_mape=("sap_eski", lambda s: s.abs().mean()), yeni_mape=("sap_yeni", lambda s: s.abs().mean()),
            plan_mape=("sap_lep", lambda s: s.abs().mean()))
print(f"Son 12 ay ({son12['date'].min():%Y-%m-%d} -> {son12['date'].max():%Y-%m-%d}), egitim ici, %:")
print(kontrol.round(2).to_string())
fark = (son12["yeni"] - son12["eski"]) / son12["eski"] * 100
print(f"\nYeni ve eski tahmin farki: ortalama {fark.mean():+.2f}%, mutlak ortalama {fark.abs().mean():.2f}%")

Son 12 ay (2025-09-23 -> 2026-09-22), egitim ici, %:
       saat  eski_sapma  yeni_sapma  plan_sapma  eski_mape  yeni_mape  plan_mape
dilim                                                                           
<20    5736       -0.80       -0.07        0.46       2.20       0.51       2.62
20-24   888       -0.21       -0.08        1.02       2.53       0.47       2.75
24-27  1296       -0.91       -0.05       -1.11       2.14       0.41       2.42
>=27    768        0.29       -0.03       -1.03       1.79       0.38       2.31

Yeni ve eski tahmin farki: ortalama +0.67%, mutlak ortalama 1.93%


## F. Kayıt (eski dosyalar arşivleniyor)

In [6]:
zaman = pd.Timestamp.now(tz="Europe/Istanbul").strftime("%Y%m%d_%H%M")
arsiv = os.path.join(KOK, "agent", "arsiv", zaman)
os.makedirs(arsiv, exist_ok=True)
for k, v in ESKI.items():
    for uz in [".json", "_meta.json"]:
        shutil.copy2(os.path.join(KOK, "agent", v["dosya"] + uz), arsiv)
for k, v in ESKI.items():
    YENI[k].save_model(os.path.join(KOK, "agent", v["dosya"] + ".json"))
    meta = dict(v["meta"])
    meta.update({"ankara_bazlari_c": [BH, BC], "surum": f"3d_uretim_{zaman}",
                 "egitim_bitis": str((KESIM - pd.Timedelta(days=1)).date()), "agac_sayisi": AGAC[k],
                 "egitim_yontemi": "3c yapisi; butun veri; agac sayisi 3c erken durdurmasindan sabit; erken durdurma yok",
                 "onceki_surum_arsivi": arsiv, "plan_duz_gunleri": [str(g.date()) for g in duz]})
    if "om_t" in meta["ozellikler"]:
        meta.update({"om_bazlari_c": [OBH, OBC], "sapma_tablosu_ay_saat_c": SAPMA})
    with open(os.path.join(KOK, "agent", v["dosya"] + "_meta.json"), "w", encoding="utf-8") as fh:
        json.dump(meta, fh, ensure_ascii=False, indent=2)
print("Eski modeller arsivlendi:", arsiv)
print("Yeni modeller kaydedildi:", [v["dosya"] for v in ESKI.values()])
print("Geri donmek icin: arsiv klasorundeki 6 dosyayi agent klasorune geri kopyalayin.")

Eski modeller arsivlendi: C:\Deep_Learning_to_AI_Agents\Bitirme_Proje_Bonus\agent\arsiv\20261002_1617
Yeni modeller kaydedildi: ['tr_xgb_gunoncesi_secilen', 'tr_xgb_gunoncesi_yedek_havasiz', 'tr_xgb_gunoncesi_yedek_plansiz']
Geri donmek icin: arsiv klasorundeki 6 dosyayi agent klasorune geri kopyalayin.
